In [ ]:
samples = [True,False,False]
all(samples)
import numpy as np
import pandas as pd
samples = np.array(['사과','바나나','딸기','사과','오렌지','바나나','바나나'])
df = pd.DataFrame(samples,columns=['과일'])
df['과일'].mode()

In [ ]:
# 결측치 탐지
import pandas as pd
import numpy as np

df = pd.read_csv(r'data\messy_data.csv')
print(f'원본 데이터 shape : { df.shape}')
print(f'결측치 개수 : { df.isnull().sum().sum()}')
print('='*50)
print(f'결측치 비율 : { df.isnull().mean().round(2)}')
print(f'결측치가 포함된 행의개수 : { df.isnull().any(axis=1).sum()}')

In [ ]:
df.tail().isnull().astype(int)

In [ ]:
# 결측치 삭제
# 결측치가 있는 행만 출력
df[df.isna().any(axis=1)]
# 결측치가 하나라도 있는 행을 삭제
df_drop_any = df.dropna(how='any')
# 모든 값이 NaN인 행만 삭제
df_drop_all = df.dropna(how='all')
len(df_drop_all) - len(df)
# 특정컬럼기준 삭제
df.dropna(subset=['age','salary'])
# thresh - 최소 N개 유효값이 있어야 보존
df_drop_thresh = df.dropna(thresh=4)
df_drop_thresh[df_drop_thresh.isna().any(axis=1)]

In [ ]:
# 결측치 대치
# 고정값으로 대치
df_fill_zero =  df.copy()
temp = df_fill_zero.isna().any()
print(temp[temp.values].index)
df_fill_zero['score'].fillna( df_fill_zero['score'].median() )
dept_mode = df_fill_zero['department'].mode()[0]
df_fill_zero['age'].bfill()

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/messy_data.csv')
df['score'].interpolate(method='polynomial',order=3)

In [ ]:
import pandas as pd
import numpy as np
from glob import glob
df = pd.read_csv(glob('../data/*.csv')[1],low_memory=False)

In [ ]:
len(df.columns)

In [74]:
# 이상치 탐지
%pwd
import pandas as pd
import numpy as np
df = pd.read_csv(r'data\messy_data.csv')
numeric_cols = df.describe().columns
# 이상치 탐지를 위해서 결측치를 중앙값으로 대처
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [75]:
# IQR 
def detect_outlier_iqr(data,column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3-Q1
    lower = Q1 - IQR*1.5
    upper = Q3 + IQR*1.5
    outlier_mask = (data[column] > upper ) | (data[column] < lower)
    return outlier_mask,lower,upper,Q1,Q3,IQR

In [76]:
numeric_cols = df.describe().columns[ 1: ]

iqr_df = df.copy()
total_removed_counts = 0
for colname in numeric_cols:    
    outlier_mask,lower,upper,Q1,Q3,IQR =  detect_outlier_iqr(iqr_df,colname)
    print(f'컬럼명 : {colname}')
    print(f'이상치 개수 : {outlier_mask.sum()}')
    print('-'*50)
    before = len(iqr_df)
    iqr_df = iqr_df[~outlier_mask]
    removed = before - len(iqr_df)
    total_removed_counts += removed

print(f'IQR 제거후 shape {iqr_df.shape}')
print(f'총 제거수 : {total_removed_counts}')



컬럼명 : age
이상치 개수 : 4
--------------------------------------------------
컬럼명 : salary
이상치 개수 : 6
--------------------------------------------------
컬럼명 : score
이상치 개수 : 0
--------------------------------------------------
IQR 제거후 shape (102, 6)
총 제거수 : 10


In [145]:
# 데이터가 정규분포를 따른다고 가정
from scipy import stats
# (각 요소 - 평균) / 표준편차
def detect_outlier_zscore(data, column, threshold=3):
    z_score = np.abs(stats.zscore(data[column].dropna()))
    return z_score > threshold

In [101]:
for col in numeric_cols:
    before = len(df)
    zscore_index = detect_outlier_zscore(df,col)
    df = df[zscore_index]
    removed = before - len(df)
    print(col, removed)

age 3
salary 3
score 0


In [103]:
# 이상치 탐색(수치형 데이터)
# 원본이아닌 복사본으로 결측치 이상치 탐색 대처
# IQR
# Z-score

In [120]:
import pandas as pd
filepath = r'C:\skn29_lecture\share_data\아파트(매매)_실거래가_20260403104248.csv'
df = pd.read_csv(filepath,encoding='cp949',header=15)
df['거래금액(만원)'] = df['거래금액(만원)'].str.replace(',','').astype(int)

In [ ]:
# 이상치가 아닌 clean area 해당하는 시군구
# 이상치로 판단되는 시군구




,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),동,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자
0,1,서울특별시 강남구 역삼동,720-25,720,25,대우디오빌,30.03,202603,31,27000,-,17,개인,개인,2002,언주로 427,-,직거래,-,-


In [143]:
outlier_index,lower,upper,Q1,Q3,IQR = detect_outlier_iqr(df,'거래금액(만원)')
df[outlier_index]['시군구'].unique() , lower, upper

(array(['서울특별시 강남구 압구정동', '서울특별시 강남구 삼성동', '서울특별시 강남구 청담동',
        '서울특별시 강남구 신사동', '서울특별시 강남구 대치동', '서울특별시 강남구 도곡동'], dtype=object),
 np.float64(-136375.0),
 np.float64(656625.0))

In [144]:
df[~outlier_index]['시군구'].unique()

array(['서울특별시 강남구 역삼동', '서울특별시 강남구 도곡동', '서울특별시 강남구 청담동', '서울특별시 강남구 수서동',
       '서울특별시 강남구 일원동', '서울특별시 강남구 개포동', '서울특별시 강남구 대치동', '서울특별시 강남구 논현동',
       '서울특별시 강남구 삼성동', '서울특별시 강남구 자곡동', '서울특별시 강남구 압구정동',
       '서울특별시 강남구 신사동', '서울특별시 강남구 세곡동', '서울특별시 강남구 율현동'], dtype=object)

In [149]:
df[detect_outlier_zscore(df,'거래금액(만원)')]['시군구'].unique()

array(['서울특별시 강남구 삼성동', '서울특별시 강남구 압구정동', '서울특별시 강남구 청담동'], dtype=object)